In [1]:
# ============================================================
# EXPERIMENT 005
# XGBoost missingness model with a higher tree limit
# ============================================================

from pathlib import Path
from time import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier


warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

EXPERIMENT_ID = "EXP-005"

RANDOM_STATE = 42
N_SPLITS = 3

TARGET = "addicted_label"
ID_COLUMN = "id"

EXP_004_OOF_AUC = 0.963290

MINIMUM_IMPROVEMENT = 0.0002
SUBMISSION_THRESHOLD = (
    EXP_004_OOF_AUC + MINIMUM_IMPROVEMENT
)


PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent


DATA_DIR = PROJECT_DIR / "data"
SUBMISSION_DIR = PROJECT_DIR / "submissions"

SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = (
    DATA_DIR / "sample_submission.csv"
)


# ============================================================
# 2. LOAD DATA
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH
)


print(f"Train shape:             {train.shape}")
print(f"Test shape:              {test.shape}")
print(
    f"Sample submission shape: "
    f"{sample_submission.shape}"
)


assert TARGET in train.columns
assert TARGET not in test.columns
assert len(test) == len(sample_submission)


# ============================================================
# 3. CREATE ORIGINAL FEATURES AND TARGET
# ============================================================

X = train.drop(
    columns=[TARGET, ID_COLUMN]
).copy()

y = train[TARGET].astype(int).copy()

X_test = test.drop(
    columns=[ID_COLUMN]
).copy()


assert list(X.columns) == list(X_test.columns)


original_features = X.columns.tolist()


# ============================================================
# 4. ADD MISSINGNESS INDICATORS
# ============================================================

columns_with_missing_values = [
    column
    for column in original_features
    if (
        X[column].isna().any()
        or X_test[column].isna().any()
    )
]


for column in columns_with_missing_values:

    indicator_name = f"{column}__missing"

    X[indicator_name] = (
        X[column]
        .isna()
        .astype("int8")
    )

    X_test[indicator_name] = (
        X_test[column]
        .isna()
        .astype("int8")
    )


missing_indicator_columns = [
    f"{column}__missing"
    for column in columns_with_missing_values
]


assert list(X.columns) == list(X_test.columns)


print(f"\nOriginal features:         {len(original_features)}")
print(
    f"Missing indicators:       "
    f"{len(missing_indicator_columns)}"
)
print(f"Total features:            {X.shape[1]}")


# ============================================================
# 5. IDENTIFY FEATURE TYPES
# ============================================================

categorical_columns = X.select_dtypes(
    include=[
        "object",
        "category",
        "bool"
    ]
).columns.tolist()


numeric_columns = X.columns.difference(
    categorical_columns
).tolist()


print(f"\nNumerical features:   {len(numeric_columns)}")
print(
    f"Categorical features: "
    f"{len(categorical_columns)}"
)

print("\nCategorical columns:")
print(categorical_columns)


# ============================================================
# 6. PREPROCESSING
# ============================================================

try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse_output=True
    )

except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse=True
    )


numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "one_hot",
            one_hot_encoder
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ],
    remainder="drop"
)


# ============================================================
# 7. XGBOOST CONFIGURATION
# ============================================================

# Only the tree limit and early-stopping patience differ
# from EXP-004.

model_parameters = {
    "objective": "binary:logistic",
    "eval_metric": "auc",

    "n_estimators": 3000,
    "learning_rate": 0.05,

    "max_depth": 6,
    "min_child_weight": 5,

    "subsample": 0.80,
    "colsample_bytree": 0.80,

    "reg_alpha": 0.0,
    "reg_lambda": 1.0,

    "tree_method": "hist",

    "early_stopping_rounds": 100,

    "random_state": RANDOM_STATE,
    "n_jobs": -1
}


# ============================================================
# 8. STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


out_of_fold_predictions = np.zeros(
    len(train),
    dtype=float
)

test_predictions = np.zeros(
    len(test),
    dtype=float
)


fold_scores = []
best_iterations = []

experiment_start = time()


for fold_number, (
    train_indices,
    validation_indices
) in enumerate(
    cv.split(X, y),
    start=1
):

    fold_start = time()

    X_train_fold = X.iloc[
        train_indices
    ]

    X_validation_fold = X.iloc[
        validation_indices
    ]

    y_train_fold = y.iloc[
        train_indices
    ]

    y_validation_fold = y.iloc[
        validation_indices
    ]


    fold_preprocessor = clone(
        preprocessor
    )


    X_train_processed = (
        fold_preprocessor.fit_transform(
            X_train_fold
        )
    )

    X_validation_processed = (
        fold_preprocessor.transform(
            X_validation_fold
        )
    )

    X_test_processed = (
        fold_preprocessor.transform(
            X_test
        )
    )


    if fold_number == 1:
        print(
            "\nProcessed training matrix shape:",
            X_train_processed.shape
        )


    model = XGBClassifier(
        **model_parameters
    )


    model.fit(
        X_train_processed,
        y_train_fold,
        eval_set=[
            (
                X_validation_processed,
                y_validation_fold
            )
        ],
        verbose=False
    )


    validation_probabilities = (
        model.predict_proba(
            X_validation_processed
        )[:, 1]
    )


    fold_test_probabilities = (
        model.predict_proba(
            X_test_processed
        )[:, 1]
    )


    out_of_fold_predictions[
        validation_indices
    ] = validation_probabilities


    test_predictions += (
        fold_test_probabilities
        / N_SPLITS
    )


    fold_auc = roc_auc_score(
        y_validation_fold,
        validation_probabilities
    )


    fold_scores.append(
        float(fold_auc)
    )


    best_iteration = getattr(
        model,
        "best_iteration",
        None
    )

    best_iterations.append(
        best_iteration
    )


    fold_minutes = (
        time() - fold_start
    ) / 60


    print(
        f"Fold {fold_number}/{N_SPLITS} | "
        f"AUC: {fold_auc:.6f} | "
        f"Best iteration: {best_iteration} | "
        f"Time: {fold_minutes:.2f} minutes"
    )


# ============================================================
# 9. VALIDATION RESULTS
# ============================================================

overall_oof_auc = roc_auc_score(
    y,
    out_of_fold_predictions
)


mean_fold_auc = float(
    np.mean(fold_scores)
)

std_fold_auc = float(
    np.std(fold_scores)
)


difference_vs_exp_004 = (
    overall_oof_auc
    - EXP_004_OOF_AUC
)


total_minutes = (
    time() - experiment_start
) / 60


print("\n" + "=" * 60)
print("EXPERIMENT 005 RESULTS")
print("=" * 60)


for fold_number, score in enumerate(
    fold_scores,
    start=1
):
    print(
        f"Fold {fold_number} AUC: "
        f"{score:.6f}"
    )


print(
    f"\nMean fold AUC:          "
    f"{mean_fold_auc:.6f}"
)

print(
    f"Fold AUC SD:            "
    f"{std_fold_auc:.6f}"
)

print(
    f"OOF AUC:                "
    f"{overall_oof_auc:.6f}"
)

print(
    f"EXP-004 OOF:            "
    f"{EXP_004_OOF_AUC:.6f}"
)

print(
    f"Difference vs EXP-004:  "
    f"{difference_vs_exp_004:+.6f}"
)

print(
    f"Best iterations:        "
    f"{best_iterations}"
)

print(
    f"Runtime:                "
    f"{total_minutes:.2f} minutes"
)


# ============================================================
# 10. INTERPRET RESULT
# ============================================================

print("\nInterpretation:")


if (
    difference_vs_exp_004
    >= MINIMUM_IMPROVEMENT
):
    print(
        "The higher tree limit produced a useful improvement. "
        "EXP-004 was likely constrained by the 1,500-tree limit."
    )

elif (
    difference_vs_exp_004
    > -MINIMUM_IMPROVEMENT
):
    print(
        "The result is effectively unchanged. More boosting "
        "rounds did not produce a meaningful improvement."
    )

else:
    print(
        "The higher tree limit reduced validation performance. "
        "Retain EXP-004 as the stronger model."
    )


if all(
    iteration is not None
    and iteration >= 2900
    for iteration in best_iterations
):
    print(
        "Warning: all folds again approached the maximum tree "
        "limit. The model may still be iteration-constrained."
    )

elif any(
    iteration is not None
    and iteration < 2900
    for iteration in best_iterations
):
    print(
        "Early stopping activated before the 3,000-tree limit "
        "for at least one fold."
    )

Train shape:             (691369, 14)
Test shape:              (296302, 13)
Sample submission shape: (296302, 2)

Original features:         12
Missing indicators:       12
Total features:            24

Numerical features:   21
Categorical features: 3

Categorical columns:
['gender', 'stress_level', 'academic_work_impact']

Processed training matrix shape: (460912, 29)
Fold 1/3 | AUC: 0.963096 | Best iteration: 2442 | Time: 1.68 minutes
Fold 2/3 | AUC: 0.963999 | Best iteration: 2376 | Time: 1.63 minutes
Fold 3/3 | AUC: 0.963887 | Best iteration: 2408 | Time: 1.66 minutes

EXPERIMENT 005 RESULTS
Fold 1 AUC: 0.963096
Fold 2 AUC: 0.963999
Fold 3 AUC: 0.963887

Mean fold AUC:          0.963661
Fold AUC SD:            0.000402
OOF AUC:                0.963659
EXP-004 OOF:            0.963290
Difference vs EXP-004:  +0.000369
Best iterations:        [2442, 2376, 2408]
Runtime:                4.97 minutes

Interpretation:
The higher tree limit produced a useful improvement. EXP-004 was like

AssertionError: 

In [2]:
# ============================================================
# 11. PREDICTION SANITY CHECKS
# ============================================================

prediction_summary = pd.Series(
    test_predictions,
    name="predicted_probability"
).describe()

print("\nTest prediction summary:")
print(prediction_summary)

print(f"\nExact minimum: {test_predictions.min():.12f}")
print(f"Exact maximum: {test_predictions.max():.12f}")

below_zero_count = int((test_predictions < 0).sum())
above_one_count = int((test_predictions > 1).sum())

print(f"Predictions below zero: {below_zero_count}")
print(f"Predictions above one:  {above_one_count}")

assert np.isfinite(out_of_fold_predictions).all()
assert np.isfinite(test_predictions).all()

# Allow only extremely small floating-point deviations.
FLOAT_TOLERANCE = 1e-6

assert test_predictions.min() >= -FLOAT_TOLERANCE, (
    "Predictions are meaningfully below zero."
)

assert test_predictions.max() <= 1 + FLOAT_TOLERANCE, (
    "Predictions are meaningfully above one."
)

# Correct tiny floating-point deviations.
test_predictions = np.clip(
    test_predictions,
    0.0,
    1.0
)

assert ((test_predictions >= 0) & (test_predictions <= 1)).all()
assert np.std(test_predictions) > 0

print("\nPredictions successfully clipped to [0, 1].")


Test prediction summary:
count    296302.000000
mean          0.709590
std           0.367851
min           0.000065
25%           0.392529
50%           0.945544
75%           0.999532
max           1.000000
Name: predicted_probability, dtype: float64

Exact minimum: 0.000065133174
Exact maximum: 1.000000029802
Predictions below zero: 0
Predictions above one:  25

Predictions successfully clipped to [0, 1].


In [3]:
# ============================================================
# 12. CONDITIONAL SUBMISSION
# ============================================================

if overall_oof_auc >= SUBMISSION_THRESHOLD:

    submission = sample_submission.copy()

    assert TARGET in submission.columns

    submission[TARGET] = test_predictions

    submission_path = (
        SUBMISSION_DIR
        / "exp_005_xgb_more_trees.csv"
    )

    submission.to_csv(
        submission_path,
        index=False
    )

    print(
        "\nSubmission created because EXP-005 improved "
        "upon EXP-004 by at least "
        f"{MINIMUM_IMPROVEMENT:.4f}."
    )

    print(f"Saved to:\n{submission_path}")
    print("\nSubmission preview:")
    print(submission.head())

else:
    submission_path = None

    print(
        "\nNo submission created because OOF AUC did not reach "
        f"{SUBMISSION_THRESHOLD:.6f}."
    )


Submission created because EXP-005 improved upon EXP-004 by at least 0.0002.
Saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\02-smartphone-addiction\submissions\exp_005_xgb_more_trees.csv

Submission preview:
       id  addicted_label
0  691369        0.999691
1  691370        0.952882
2  691371        0.932459
3  691372        0.990339
4  691373        0.998382


### Experiment Log

In [4]:
from datetime import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd


EXPERIMENT_LOG_PATH = PROJECT_DIR / "experiment_log.csv"


def log_experiment(
    experiment_id,
    description,
    model,
    features,
    validation_method,
    cv_scores,
    kaggle_score=None,
    changes="",
    submission_file="",
    notes="",
    log_path=EXPERIMENT_LOG_PATH
):
    """
    Add or update one experiment in experiment_log.csv.

    If the experiment_id already exists, its previous row is replaced.
    """

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    cv_scores = [float(score) for score in cv_scores]

    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))

    kaggle_score_value = (
        float(kaggle_score)
        if kaggle_score is not None
        else np.nan
    )

    kaggle_cv_gap = (
        kaggle_score_value - cv_mean
        if pd.notna(kaggle_score_value)
        else np.nan
    )

    experiment_record = {
        "experiment_id": experiment_id,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "description": description,
        "model": model,
        "features": json.dumps(list(features)),
        "n_features": len(features),
        "validation_method": validation_method,
        "cv_scores": json.dumps(cv_scores),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "kaggle_score": kaggle_score_value,
        "kaggle_cv_gap": kaggle_cv_gap,
        "changes": changes,
        "submission_file": submission_file,
        "notes": notes
    }

    if log_path.exists():
        experiments = pd.read_csv(log_path)

        # Prevent duplicate rows when rerunning the same experiment cell.
        if "experiment_id" in experiments.columns:
            experiments = experiments[
                experiments["experiment_id"] != experiment_id
            ].copy()
    else:
        experiments = pd.DataFrame()

    new_row = pd.DataFrame([experiment_record])

    experiments = pd.concat(
        [experiments, new_row],
        ignore_index=True
    )

    experiments = experiments.sort_values(
        by="experiment_id"
    ).reset_index(drop=True)

    experiments.to_csv(log_path, index=False)

    print(f"Logged {experiment_id}")
    print(f"CV mean:       {cv_mean:.6f}")
    print(f"CV SD:         {cv_std:.6f}")

    if pd.notna(kaggle_score_value):
        print(f"Kaggle score:  {kaggle_score_value:.6f}")
        print(f"Kaggle-CV gap: {kaggle_cv_gap:+.6f}")

    print(f"Log saved to:  {log_path}")

    return experiments

In [5]:
experiments = log_experiment(
    experiment_id="EXP-005",
    description=(
        "XGBoost missingness-indicator model with the estimator limit "
        "increased from 1,500 to 3,000 boosting rounds."
    ),
    model="XGBClassifier",
    features=X.columns.tolist(),
    validation_method="3-fold StratifiedKFold with ROC AUC",
    cv_scores=[
        0.963096,
        0.963999,
        0.963887
    ],
    kaggle_score=0.96534,
    changes=(
        "Retained all EXP-004 features, missingness indicators, preprocessing, "
        "folds, and XGBoost parameters. Increased n_estimators from 1500 to "
        "3000 and early_stopping_rounds from 75 to 100."
    ),
    submission_file="exp_005_xgb_more_trees.csv",
    notes=(
        "OOF AUC was 0.963659 with fold SD 0.000402, improving upon EXP-004 "
        "by 0.000369. Kaggle improved from 0.964750 to 0.965340, a gain of "
        "0.000590. Best iterations were 2442, 2376, and 2408, confirming that "
        "the previous 1500-tree limit constrained the model. Tiny floating-point "
        "probability overruns were clipped to the valid [0, 1] range before "
        "submission."
    )
)

experiments.tail()

Logged EXP-005
CV mean:       0.963661
CV SD:         0.000402
Kaggle score:  0.965340
Kaggle-CV gap: +0.001679
Log saved to:  C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\02-smartphone-addiction\experiment_log.csv


,experiment_id,timestamp,description,model,features,n_features,validation_method,cv_scores,cv_mean,cv_std,kaggle_score,kaggle_cv_gap,changes,submission_file,notes
0,EXP-001,2026-08-02 21:49:55,Initial XGBoost baseline using median-imputed ...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.962477, 0.963382, 0.963244]",0.963034,0.000398,0.96449,0.001456,Established the first end-to-end baseline usin...,exp_001_xgb_baseline.csv,OOF AUC was 0.963034 with fold SD 0.000398. Ka...
1,EXP-002,2026-08-02 21:53:06,Logistic-regression control using standardized...,LogisticRegression,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.910347, 0.911866, 0.912106]",0.911440,0.000779,NaN,NaN,Replaced the XGBoost baseline with a regulariz...,NaN,OOF AUC was 0.911437 with fold SD 0.000779. Th...
2,EXP-003,2026-08-02 22:11:48,CatBoost comparison using native numerical mis...,CatBoostClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.961586, 0.962336, 0.962426]",0.962116,0.000377,NaN,NaN,Replaced XGBoost and one-hot preprocessing wit...,NaN,OOF AUC was 0.962115 with fold SD 0.000376. Ca...
3,EXP-004,2026-08-02 22:18:27,XGBoost baseline extended with binary missingn...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",24,3-fold StratifiedKFold with ROC AUC,"[0.962723, 0.963675, 0.963473]",0.963290,0.000410,0.96475,0.001460,Added one binary missingness indicator for eac...,exp_004_xgb_missing_indicators.csv,OOF AUC was 0.963290 with fold SD 0.000409. Th...
4,EXP-005,2026-08-02 22:27:47,XGBoost missingness-indicator model with the e...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",24,3-fold StratifiedKFold with ROC AUC,"[0.963096, 0.963999, 0.963887]",0.963661,0.000402,0.96534,0.001679,"Retained all EXP-004 features, missingness ind...",exp_005_xgb_more_trees.csv,"OOF AUC was 0.963659 with fold SD 0.000402, im..."
